In [ ]:
%%capture
!pip install tweepy

In [ ]:
import os
import tweepy
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from typing import Optional, Union

from collections import Counter
import ast
import time

In [ ]:
# Configuration: Authenticate with Twitter API v2
# Set TWITTER_BEARER_TOKEN in your environment or enter it below:
BEARER_TOKEN = os.environ.get("TWITTER_BEARER_TOKEN", "YOUR_BEARER_TOKEN_HERE")

# Create Tweepy client
api = tweepy.Client(bearer_token=BEARER_TOKEN)


# Twitter Scraper

In [ ]:
def check_user(username: str) -> bool:
        """
        Check if a user exists
        :param screen_name:
        :return: True if the user exists
        """
        try:
            # NB : pass keyword argument `screen_name` to get_user()
            twitter_user = api.get_users(usernames=username)
            if twitter_user.data:
                return True
            else:
                return False
        except Exception as e:
            print("Error: ", e)
            return False

        
def tweet_scraper(twitter_user_name, file_path):
    # Define the username you want to collect tweets from
    user_name = twitter_user_name

    twitter_user = api.get_user(username=user_name)

    # Define an empty list to store all the collected tweets
    all_tweets = []

    # Define the maximum number of tweets you want to collect (set to 1000 for this example)
    max_tweets = 3500

    # Define the number of tweets to retrieve in each API call
    batch_size = 100

    # Define a variable to keep track of the oldest tweet ID we've collected so far
    oldest_tweet_id = None
    
    try:
        # Loop until we've collected the maximum number of tweets, or there are no more tweets to collect
        while len(all_tweets) < max_tweets:
            # Determine the number of tweets we want to collect in this batch (minimum of batch_size and the remaining number of tweets to collect)
            num_tweets = min(batch_size, max_tweets - len(all_tweets))

            # Use the API's get_users_tweets method to get the most recent tweets from the user's timeline, starting from the oldest tweet ID we've collected so far
            tweets = api.get_users_tweets(twitter_user.data.id, 
                                          expansions="author_id",
                                          tweet_fields=['context_annotations','created_at','geo', 'public_metrics'],
                                          user_fields=['id', 'username', 'description', 'public_metrics'],
                                          max_results=num_tweets, 
                                          until_id=oldest_tweet_id)
            time.sleep(1)
            # If there are no more tweets to collect, break out of the loop
            if not tweets.data:
                break
                
            user_dict = {user['id']: user for user in tweets.includes['users']}
            user_info = user_dict.get(twitter_user.data.id)
            
            
            # Add the collected tweets to the list of all tweets
            all_tweets.extend(tweets.data)

            # Update the oldest_tweet_id variable to the ID of the oldest tweet we've collected so far
            oldest_tweet_id = min(tweet.id for tweet in tweets.data)

            # Print out a status update to show how many tweets we've collected so far
            print(f"Collected {len(all_tweets)} tweets so far...")

        # Loop through each tweet in the list of all tweets and print out the tweet's text
        print(len(all_tweets))

        out_tweets = [[tweet.id, tweet.created_at, tweet.text, tweet.public_metrics['like_count'], tweet.public_metrics['retweet_count'], 
                       tweet.public_metrics['quote_count'],tweet.public_metrics['reply_count'], 
                       tweet.source, user_info['id'], user_info['username'],
                       user_info['description'], user_info['public_metrics']['followers_count'],
                       user_info['public_metrics']['following_count'], user_info['public_metrics']['listed_count'], 
                       user_info['public_metrics']['tweet_count']] for tweet in all_tweets]

        # Create a dataframe from the list of tweets
        df = pd.DataFrame(out_tweets, columns=['twitter_id', 'date', 'tweet_text', 'tweet_favorite_count',
                                       'tweet_retweet_count', 'tweet_quote_count', 'tweet_reply_count',
                                        'tweet_source', 'user_id', 'user_screen_name', 'user_description', 
                                        'user_followers_count', 'user_friends_count' ,'user_listed_count',
                                        'user_statuses_count'])
        
        df.to_csv(os.path.join(file_path, "%s.csv" % user_name), index=False)
    except Exception as e:
        print("Error", e)

In [ ]:
# Create a TwitterScraper object for tweepy
# twitter_scraper = TwitterScraper(consumer_key=CONSUMER_KEY,
#                                  consumer_secret=CONSUMER_SECRET,
#                                  access_key=ACCESS_KEY,
#                                  access_secret=ACCESS_SECRET,
#                                  bearer_token=BEARER_TOKEN,
#                                  file_path=DATA_PATH)

In [ ]:
# # check a Twitter username
# twitter_scraper.check_user(username='LatexKaktus')

# # check a Twitter user id
# # twitter_scraper.fetch_user_id(screen_name='elliot20201')
# %%time
# tweet_scraper("LatexKaktus")

In [ ]:
# https://proclusacademy.com/blog/stratified_sampling_pandas/

df = pd.read_csv("/kaggle/input/user-handle-list/user_handles.csv")

df_filtered = df[~df['unique_handles'].apply(lambda x: len(ast.literal_eval(x)) == 0)]
print(df_filtered['diagnosis'].value_counts())


df_diagnosed = df_filtered.loc[df_filtered['user_group'] == 'diagnosed']

stratified_diagnosed = df_diagnosed.groupby('diagnosis').apply(
    lambda x: x.sample(frac=0.10, random_state = 12)
)

stratified_diagnosed = stratified_diagnosed.droplevel(0)
print(stratified_diagnosed.shape)
stratified_diagnosed['diagnosis'].value_counts()

In [ ]:
%%time
output_path = '/kaggle/working/'
invalid_users = []

data = stratified_diagnosed[100:]
cnt = 0
for index, row in data.iterrows():
    handles_dict = ast.literal_eval(row.handles_count)
    
    c = Counter(handles_dict)

    most_interacted = c.most_common(5)
    most_interacted_list = [user[0] for user in most_interacted]
    
    user_found = 0 
    
    if most_interacted_list:
        if not os.path.exists(output_path + str(row.userfile_name)):
            os.makedirs(output_path + str(row.userfile_name))
            print("Made directory......")
            for user in most_interacted_list:
                file_path = output_path + str(row.userfile_name) + "/"
                
                try:
                    if check_user(username=user):
                        tweet_scraper(user, file_path)
                        print("-------Saved Tweets for: ", user, "-------")
                        user_found += 1
                    else:
                        print("-------User : ", user, " not found-------")
                except Exception as e:
                    print("Error", e)
    
        print("-------Total scraped ", user_found, " among", len(most_interacted_list), " users-------")
        cnt += 1
        print("-------Collection ego-network for total: ", cnt, " user, last user: ", str(row.userfile_name), "-------")
    else:
        invalid_users.append(row.userfile_name)
    
#     print("username: ", row.user_screen_name, "user group: ", row.user_group, "most interacted users: ", most_interacted, "total handles mentioned: ", len(row.unique_handles))


In [ ]:
# os.remove("/kaggle/working/usuario_15000")
# import shutil
# shutil.rmtree("/kaggle/working/usuario_15000")

tweet_id -> id_str

day, time -> created_at

tweet_text	full_text

tweet_favorite_count -> favorite_count

tweet_retweet_count -> retweet_count

tweet_quote_count -> quote_count

tweet_reply_count -> reply_count

tweet_source -> source

user_id -> user.id_str

user_screen_name -> user.screen_name

user_description -> user.description

user_followers_count -> user.followers_count

user_friends_count -> user.friends_count

user_listed_count -> user.listed_count

user_statuses_count -> user.statuses_count





In [ ]:
#  df = pd.DataFrame(out_tweets, columns=['twitter_id', 'date', 'tweet_text', 'tweet_favorite_count',
#                                        'tweet_retweet_count', 'tweet_quote_count', 'tweet_reply_count',
#                                         'tweet_source', 'user_id', 'user_screen_name', 'user_description', 
#                                         'user_followers_count', 'user_friends_count' ,'user_listed_count',
#                                         'user_statuses_count'])
                   
                   
                   
# out_tweets = [[tweet.id_str, tweet.created_at, tweet.full_text.encode("utf-8"), tweet.favorite_count, tweet.retweet_count, tweet.quote_count, tweet.reply_count, tweet.source, tweet.user.id_str, tweet.user.screen_name, tweet.user.description, tweet.user.followers_count, tweet.user.friends_count, tweet.user.listed_count, tweet.user.statuses_count] for tweet in alltweets]

# https://developer.twitter.com/en/docs/twitter-api/data-dictionary/object-model/user
# https://developer.twitter.com/en/docs/twitter-api/data-dictionary/object-model/tweet
